# 00. Kiểm tra dữ liệu và chia tập

Đọc CSV gốc, kiểm tra chất lượng và lưu một bộ chia dùng chung cho nhóm.

Nguồn dữ liệu là [UCI 697 — Predict Students’ Dropout and Academic Success](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success), Realinho, Vieira Martins, Machado và Baptista (2021), [DOI 10.24432/C5MC89](https://doi.org/10.24432/C5MC89), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Đầu ra** là `data/splits/splits.json`. Các chỉ số trong file tính từ 0, theo thứ tự dòng của CSV sau tiêu đề.

## Mở dữ liệu

Chọn kernel Python đã cài các thư viện trong `requirements.txt`.

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Tìm thư mục gốc để notebook đọc đúng file, kể cả khi kernel chạy từ thư mục con.
current = Path.cwd().resolve()
root = next(
    (folder for folder in (current, *current.parents)
     if (folder / "common/preprocessing.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Mở thư mục student-dropout-prediction trong VS Code rồi chạy notebook.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from common.preprocessing import (
    BINARY_COLUMNS, CATEGORICAL_COLUMNS, FEATURE_COLUMNS, LABELS, NUMERIC_COLUMNS, TARGET,
)

data_path = root / "data/raw/data.csv"
split_path = root / "data/splits/splits.json"
# File UCI ngăn cột bằng dấu chấm phẩy. Xóa khoảng trắng thừa quanh tên cột.
data = pd.read_csv(data_path, sep=";", encoding="utf-8-sig")
data.columns = data.columns.str.strip()

# Thống nhất ký tự xuống dòng trước khi tính mã nhận diện CSV trên Windows và Linux.
csv_bytes = data_path.read_bytes().replace(b"\r\n", b"\n")
data_hash = hashlib.sha256(csv_bytes).hexdigest()

## 1. Kiểm tra dữ liệu gốc

Đối chiếu đủ 36 đặc trưng, ba lớp và các cột 0/1. Các thống kê toàn tập ở đây phục vụ kiểm tra đầu vào.

In [2]:
# So tên cột trong CSV với danh sách đặc trưng và nhãn đã khai báo.
expected = set(FEATURE_COLUMNS) | {TARGET}
missing_columns = expected - set(data.columns)
unexpected_columns = set(data.columns) - expected
if missing_columns or unexpected_columns or not data.columns.is_unique:
    raise ValueError(
        f"Tên cột không khớp. Thiếu: {sorted(missing_columns)}. "
        f"Cột lạ: {sorted(unexpected_columns)}."
    )

if data[TARGET].isna().any() or set(data[TARGET]) != set(LABELS):
    raise ValueError("Target phải có đủ ba nhãn và không được để trống.")

invalid_binary = [
    column for column in BINARY_COLUMNS
    if not data[column].dropna().isin([0, 1]).all()
]
if invalid_binary:
    raise ValueError(f"Cột nhị phân có giá trị ngoài 0/1: {invalid_binary}")

missing_values = data.isna().sum()
audit = pd.Series({
    "Số sinh viên": len(data),
    "Số đặc trưng": len(FEATURE_COLUMNS),
    "Số ô thiếu": int(missing_values.sum()),
    "Số dòng trùng hoàn toàn": int(data.duplicated().sum()),
    "Số cột số": len(NUMERIC_COLUMNS),
    "Số cột phân loại": len(CATEGORICAL_COLUMNS),
    "Số cột nhị phân": len(BINARY_COLUMNS),
}, name="Kiểm tra CSV")
display(audit)
if missing_values.any():
    display(missing_values[missing_values > 0].rename("Số ô thiếu"))
display(data[["Age at enrollment", "Course", "Scholarship holder", TARGET]].head(3))

Số sinh viên               4424
Số đặc trưng                 36
Số ô thiếu                    0
Số dòng trùng hoàn toàn       0
Số cột số                    19
Số cột phân loại              9
Số cột nhị phân               8
Name: Kiểm tra CSV, dtype: int64

,Age at enrollment,Course,Scholarship holder,Target
0,20,171,0,Dropout
1,19,9254,0,Graduate
2,19,9070,0,Dropout


## 2. Chia Train / Validation / Test

Tách Test 15% trước. Sau đó tách 15/85 của phần còn lại để Validation chiếm khoảng 15% toàn bộ dữ liệu. Cả hai lần đều stratify theo nhãn.

In [3]:
from sklearn.model_selection import train_test_split

SEED = 42
TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

# Nếu nhóm đã lưu bộ chia, kiểm tra và dùng lại đúng bộ đó.
has_saved_split = split_path.is_file()
if has_saved_split:
    split = json.loads(split_path.read_text(encoding="utf-8"))
    expected_settings = (SEED, TEST_SIZE, VALIDATION_SIZE)
    saved_settings = (split["seed"], split["test_size"], split["validation_size"])
    if split["sha256"] != data_hash or split["n_rows"] != len(data):
        raise ValueError("CSV khác phiên bản đã dùng để tạo bộ chia.")
    if saved_settings != expected_settings:
        raise ValueError("Cấu hình chia tập khác bản đã lưu của nhóm.")
    print("Đọc lại bộ chia đã lưu.")
else:
    # Chia theo chỉ số dòng để sau này lấy lại đúng sinh viên từ CSV gốc.
    row_ids = np.arange(len(data))
    # Tách 15% toàn bộ dữ liệu ra làm Test.
    remaining, test = train_test_split(
        row_ids, test_size=TEST_SIZE, random_state=SEED,
        stratify=data[TARGET],
    )
    # Lấy 15/85 phần còn lại để Validation chiếm khoảng 15% tổng dữ liệu.
    train, validation = train_test_split(
        remaining, test_size=VALIDATION_SIZE / (1 - TEST_SIZE),
        random_state=SEED, stratify=data.iloc[remaining][TARGET],
    )
    split = {
        "sha256": data_hash, "n_rows": len(data), "seed": SEED,
        "test_size": TEST_SIZE, "validation_size": VALIDATION_SIZE,
        "train": train.tolist(), "validation": validation.tolist(), "test": test.tolist(),
    }

Đọc lại bộ chia đã lưu.


## 3. Tạo 5 fold dùng chung

Năm fold nằm bên trong Train. Mỗi lượt có bốn phần để fit và một phần để chấm. Các chỉ số lưu dưới dạng dòng CSV gốc.

In [4]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
if not has_saved_split:
    train_rows = np.asarray(split["train"])
    train_labels = data.iloc[train_rows][TARGET]
    splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    folds = []
    for fit_positions, score_positions in splitter.split(train_rows, train_labels):
        # Bộ chia trả vị trí trong Train. Tra train_rows để lưu chỉ số dòng CSV gốc.
        folds.append({
            "train": train_rows[fit_positions].tolist(),
            "validation": train_rows[score_positions].tolist(),
        })
    split["folds"] = folds

## 4. Kiểm tra và lưu bộ chia

Ba tập phải tách rời và bao phủ CSV. Trong mỗi fold, hai phần phải tách rời và hợp lại đúng Train. Qua năm lượt, mỗi dòng Train được dùng để chấm đúng một lần.

In [5]:
def checked_ids(indices, name):
    if not indices or any(type(i) is not int or not 0 <= i < len(data) for i in indices):
        raise ValueError(f"Chỉ số dòng không hợp lệ ở {name}.")
    unique = set(indices)
    if len(unique) != len(indices):
        raise ValueError(f"Có chỉ số dòng trùng ở {name}.")
    return unique

names = ("train", "validation", "test")
partitions = [checked_ids(split[name], name) for name in names]
# Hợp các tập phải đủ mọi dòng. Nếu tổng số dòng lớn hơn hợp thì có tập bị chồng lên nhau.
all_rows = set.union(*partitions)
if all_rows != set(range(len(data))) or sum(map(len, partitions)) != len(all_rows):
    raise ValueError("Train, Validation và Test phải tách rời và bao phủ CSV.")

if len(split["folds"]) != N_FOLDS:
    raise ValueError("Cần đúng 5 fold bên trong Train.")
# Ghi nhận dòng đã dùng để chấm để kiểm tra mỗi dòng Train chỉ được chấm một lượt CV.
scored_rows = set()
fold_summary = []
for number, fold in enumerate(split["folds"], start=1):
    fit_rows = checked_ids(fold["train"], f"fold {number} train")
    score_rows = checked_ids(fold["validation"], f"fold {number} validation")
    if fit_rows & score_rows or (fit_rows | score_rows) != partitions[0]:
        raise ValueError(f"Fold {number} không chia đúng các dòng của Train.")
    if scored_rows & score_rows:
        raise ValueError("Có dòng Train được dùng để chấm ở nhiều fold.")
    scored_rows.update(score_rows)
    fold_summary.append({"Fold": number, "Số mẫu fit": len(fit_rows),
                         "Số mẫu chấm": len(score_rows)})
if scored_rows != partitions[0]:
    raise ValueError("Các phần chấm chưa bao phủ Train.")

if not has_saved_split:
    split_path.parent.mkdir(parents=True, exist_ok=True)
    with split_path.open("x", encoding="utf-8", newline="\n") as output:
        json.dump(split, output, indent=2)
        output.write("\n")
print("Bộ chia đã kiểm tra:", split_path.relative_to(root))

Bộ chia đã kiểm tra: data\splits\splits.json


## 5. Xem số mẫu và tỉ lệ lớp

Validation 15% là tập giữ riêng. Phần chấm của mỗi CV fold là một phần nhỏ nằm trong Train.

In [6]:
split_counts = pd.DataFrame([
    data.iloc[split[name]][TARGET].value_counts().reindex(LABELS, fill_value=0)
    for name in names
], index=names)
totals = split_counts.sum(axis=1)
# Chia số mẫu của từng lớp cho tổng số mẫu ngay trong tập đó.
proportions = split_counts.div(totals, axis=0).mul(100).round(2)
display(split_counts.assign(total=totals))
display(proportions.add_suffix(" (%)"))
display(pd.DataFrame(fold_summary).set_index("Fold"))

Target,Dropout,Enrolled,Graduate,total
train,995,556,1545,3096
validation,213,119,332,664
test,213,119,332,664


Target,Dropout (%),Enrolled (%),Graduate (%)
train,32.14,17.96,49.9
validation,32.08,17.92,50.0
test,32.08,17.92,50.0


,Số mẫu fit,Số mẫu chấm
Fold,,
1,2476,620
2,2477,619
3,2477,619
4,2477,619
5,2477,619


## Bàn giao

Chạy `01_eda_train.ipynb` sau khi bộ chia đã được lưu. Logistic và RF sau này dùng cùng danh sách `folds` để chọn cấu hình.